# Autoresearch Control Plane Demo

This notebook is a deterministic smoke test for the `autoresearch` control plane.

It uses the local Ollama worker for the code-edit step, but it replaces the expensive training run with a synthetic `run.log` writer so the test can finish quickly and reliably.

It covers:

1. locating the `claw-code` repo and `nodes/autoresearch-macos`
2. verifying Ollama reachability
3. running `autoresearch setup` and `autoresearch status`
4. creating or switching to an isolated `autoresearch/<tag>` branch
5. recording a synthetic baseline
6. running one candidate packet through the local worker
7. finalizing the candidate with `keep` or `discard`
8. running a short automatic loop

This proves control-plane wiring. It is not a benchmark run.

In [1]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
import urllib.error
import urllib.request
from datetime import datetime
from pathlib import Path

def is_claw_root(candidate: Path) -> bool:
    return (candidate / "src" / "main.py").exists() and (candidate / "tests" / "test_porting_workspace.py").exists()

def find_claw_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if is_claw_root(candidate):
            return candidate
        for nested in (candidate / "claw-code", candidate / "harness" / "claw-code"):
            if is_claw_root(nested):
                return nested
    raise RuntimeError(f"Could not locate claw-code root from {current}")

CLAW_ROOT = find_claw_root()
WORKSPACE_ROOT = CLAW_ROOT.parent.parent
AUTORESEARCH_ROOT = WORKSPACE_ROOT / "nodes" / "autoresearch-macos"
NOTEBOOK_DIR = CLAW_ROOT / "notebooks"
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
PYTHON = sys.executable
BRANCH_NAME = f"autoresearch/notebook-{datetime.now().strftime('%b%d-%H%M').lower()}"

def run_cmd(args: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    cwd = cwd or CLAW_ROOT
    print("$", " ".join(shlex.quote(arg) for arg in args))
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        details = result.stderr.strip() or result.stdout.strip() or "no output"
        raise RuntimeError(f"command failed with exit code {result.returncode}: {details}")
    return result

def run_json(args: list[str], cwd: Path | None = None, check: bool = True):
    result = run_cmd(args, cwd=cwd, check=check)
    return json.loads(result.stdout)

def ollama_ready(host: str) -> bool:
    try:
        request = urllib.request.Request(host.rstrip("/") + "/api/tags")
        with urllib.request.urlopen(request, timeout=3) as response:
            return 200 <= response.status < 300
    except (urllib.error.URLError, TimeoutError):
        return False

def make_log_command(val_bpb: float) -> str:
    log_text = (
        "---\n"
        f"val_bpb:          {val_bpb:.6f}\n"
        "training_seconds: 300.0\n"
        "total_seconds:    320.0\n"
        "peak_vram_mb:     1024.0\n"
        "mfu_percent:      10.00\n"
        "total_tokens_M:   1.0\n"
        "num_steps:        1\n"
        "num_params_M:     1.0\n"
        "depth:            1\n"
    )
    py = f"from pathlib import Path; Path('run.log').write_text({log_text!r}, encoding='utf-8')"
    return f"python3 -c {shlex.quote(py)}"

print({
    "claw_root": str(CLAW_ROOT),
    "autoresearch_root": str(AUTORESEARCH_ROOT),
    "python": PYTHON,
    "branch_name": BRANCH_NAME,
})

{'claw_root': '/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code', 'autoresearch_root': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos', 'python': '/Users/wongdowling/opt/anaconda3/bin/python', 'branch_name': 'autoresearch/notebook-apr08-1252'}


In [2]:
OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "qwen2.5-coder:7b"
OLLAMA_IS_READY = ollama_ready(OLLAMA_HOST)
print({"ollama_host": OLLAMA_HOST, "ollama_model": OLLAMA_MODEL, "ollama_ready": OLLAMA_IS_READY})
if not OLLAMA_IS_READY:
    raise RuntimeError("Ollama is not reachable. Start Ollama before running this notebook.")

{'ollama_host': 'http://localhost:11434', 'ollama_model': 'qwen2.5-coder:7b', 'ollama_ready': True}


In [3]:
BASELINE_COMMAND = make_log_command(0.999000)
CANDIDATE_COMMAND = make_log_command(0.998000)

smoke_packet = {
    "objective": "Modify train.py by incrementing or inserting a harmless smoke-test comment counter near the top of the file. You must change train.py on every run. A no-op is failure.",
    "description": "Increment the smoke-test comment counter in train.py.",
    "train_command": CANDIDATE_COMMAND,
    "timeout_seconds": 30,
    "log_path": "run.log",
    "results_tsv": "results.tsv",
    "syntax_check_command": "python3 -m py_compile train.py"
}
packet_path = NOTEBOOK_DIR / "autoresearch_smoke_packet.json"
packet_path.write_text(json.dumps(smoke_packet, indent=2), encoding="utf-8")
packet_path, smoke_packet

(PosixPath('/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/notebooks/autoresearch_smoke_packet.json'),
 {'objective': 'Modify train.py by incrementing or inserting a harmless smoke-test comment counter near the top of the file. You must change train.py on every run. A no-op is failure.',
  'description': 'Increment the smoke-test comment counter in train.py.',
  'train_command': 'python3 -c \'from pathlib import Path; Path(\'"\'"\'run.log\'"\'"\').write_text(\'"\'"\'---\\nval_bpb:          0.998000\\ntraining_seconds: 300.0\\ntotal_seconds:    320.0\\npeak_vram_mb:     1024.0\\nmfu_percent:      10.00\\ntotal_tokens_M:   1.0\\nnum_steps:        1\\nnum_params_M:     1.0\\ndepth:            1\\n\'"\'"\', encoding=\'"\'"\'utf-8\'"\'"\')\'',
  'timeout_seconds': 30,
  'log_path': 'run.log',
  'results_tsv': 'results.tsv',
  'syntax_check_command': 'python3 -m py_compile train.py'})

In [4]:
setup_result = run_json([
    PYTHON,
    "-m",
    "src.main",
    "autoresearch",
    "setup",
    "--root",
    str(AUTORESEARCH_ROOT),
])
status_before = run_json([
    PYTHON,
    "-m",
    "src.main",
    "autoresearch",
    "status",
    "--root",
    str(AUTORESEARCH_ROOT),
])
setup_result, status_before["isolated_branch"], status_before["has_pending_experiment"]

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main autoresearch setup --root /Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos
{
  "root": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos",
  "branch": "autoresearch/notebook-apr08-1249-test",
  "data_ready": true,
  "tokenizer_ready": true,
  "results_tsv_path": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/results.tsv",
  "results_tsv_initialized": false,
  "cache_dir": "/Users/wongdowling/.cache/autoresearch",
  "suggested_tag": "apr08"
}

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main autoresearch status --root /Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos
{
  "setup": {
    "root": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos",
    "branch": "autoresearch/notebook-apr08-1249-test",
    "data_ready": true,
    "tokenizer_ready": true,
    "results_tsv_path": "/Users/wongdowli

({'root': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos',
  'branch': 'autoresearch/notebook-apr08-1249-test',
  'data_ready': True,
  'tokenizer_ready': True,
  'results_tsv_path': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/results.tsv',
  'results_tsv_initialized': False,
  'cache_dir': '/Users/wongdowling/.cache/autoresearch',
  'suggested_tag': 'apr08'},
 True,
 False)

In [5]:
isolate_result = run_json([
    PYTHON,
    "-m",
    "src.main",
    "autoresearch",
    "isolate",
    "--root",
    str(AUTORESEARCH_ROOT),
    "--branch",
    BRANCH_NAME,
    "--from-ref",
    "master",
    "--create",
])

# Reset train.py to the new branch HEAD before the worker runs.
#
# Root cause of the "Worker did not produce a pending experiment" error:
# _collect_changed_files() detects mutations via:
#   changed = (after_paths - before_paths) | edited_paths
# where before_paths / after_paths are sets from `git status --porcelain`.
#
# If train.py was already dirty when the worker starts (left over from a
# previous run), it sits in both sets, so the set difference is empty.
# The fallback edit uses run_shell_command (not edit_file), so edited_paths
# is also empty.  Result: changed_files == [] even though the worker did edit
# the file.
#
# Resetting train.py here ensures it is clean when the worker starts, so the
# modification is always captured in after_paths - before_paths.
import subprocess as _sp
_reset = _sp.run(
    ["git", "checkout", "--", "train.py"],
    cwd=str(AUTORESEARCH_ROOT), capture_output=True, text=True
)
print("train.py reset:", "ok" if _reset.returncode == 0 else _reset.stderr.strip())

baseline_result = run_json([
    PYTHON,
    "-m",
    "src.main",
    "autoresearch",
    "baseline",
    "--root",
    str(AUTORESEARCH_ROOT),
    "--command",
    BASELINE_COMMAND,
    "--timeout-seconds",
    "30",
])
isolate_result, baseline_result.get("baseline_created"), baseline_result.get("reason")


$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main autoresearch isolate --root /Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos --branch autoresearch/notebook-apr08-1252 --from-ref master --create
{
  "root": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos",
  "previous_branch": "autoresearch/notebook-apr08-1249-test",
  "current_branch": "autoresearch/notebook-apr08-1252",
  "isolated_branch": true,
  "created": true,
  "suggested_branch": "autoresearch/apr08"
}

train.py reset: ok
$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main autoresearch baseline --root /Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos --command 'python3 -c '"'"'from pathlib import Path; Path('"'"'"'"'"'"'"'"'run.log'"'"'"'"'"'"'"'"').write_text('"'"'"'"'"'"'"'"'---\nval_bpb:          0.999000\ntraining_seconds: 300.0\ntotal_seconds:    320.0\npeak_vram_mb:     1024.0\nmfu_percent:      10.00\ntotal_tokens_M:   1.0\nnum

({'root': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos',
  'previous_branch': 'autoresearch/notebook-apr08-1249-test',
  'current_branch': 'autoresearch/notebook-apr08-1252',
  'isolated_branch': True,
  'created': True,
  'suggested_branch': 'autoresearch/apr08'},
 False,
 'baseline already recorded')

In [6]:
run_result = run_json([
    PYTHON,
    "-m",
    "src.main",
    "autoresearch",
    "run",
    "--root",
    str(AUTORESEARCH_ROOT),
    "--packet",
    str(packet_path),
    "--model",
    OLLAMA_MODEL,
    "--host",
    OLLAMA_HOST,
    "--trace",
])
status_after_run = run_json([
    PYTHON,
    "-m",
    "src.main",
    "autoresearch",
    "status",
    "--root",
    str(AUTORESEARCH_ROOT),
])
print("recommended_status:", run_result["recommended_status"])
print("tool_calls:", run_result["worker"].get("last_result", {}).get("tool_calls", []))
print("changed_files:", run_result["worker"].get("last_result", {}).get("changed_files", []))
print("pending_experiment:", status_after_run["has_pending_experiment"])
if not status_after_run["has_pending_experiment"]:
    raise RuntimeError(
        "Worker did not produce a pending experiment. Check run_result['worker']['last_result'] for why train.py was not edited."
    )
if "train.py" not in run_result["worker"].get("last_result", {}).get("changed_files", []):
    raise RuntimeError("Smoke test expected train.py to be changed, but it was not recorded in changed_files.")
run_result["recommended_status"], status_after_run["has_pending_experiment"]

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main autoresearch run --root /Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos --packet /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/notebooks/autoresearch_smoke_packet.json --model qwen2.5-coder:7b --host http://localhost:11434 --trace
{
  "setup": {
    "root": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos",
    "branch": "autoresearch/notebook-apr08-1252",
    "data_ready": true,
    "tokenizer_ready": true,
    "results_tsv_path": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/results.tsv",
    "results_tsv_initialized": false,
    "cache_dir": "/Users/wongdowling/.cache/autoresearch",
    "suggested_tag": "apr08"
  },
  "state": {
    "root": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos",
    "branch": "autoresearch/notebook-apr08-1252",
    "baseline_commit": "537c6e6",
    "baseline_bpb": 0.

('keep', True)

In [7]:
decision_command = "keep" if run_result["recommended_status"] == "keep" else "discard"
decision_result = run_json([
    PYTHON,
    "-m",
    "src.main",
    "autoresearch",
    decision_command,
    "--root",
    str(AUTORESEARCH_ROOT),
])
status_after_decision = run_json([
    PYTHON,
    "-m",
    "src.main",
    "autoresearch",
    "status",
    "--root",
    str(AUTORESEARCH_ROOT),
])
print("decision:", decision_result["decision"])
print("pending_experiment:", status_after_decision["has_pending_experiment"])
decision_result["decision"], status_after_decision["has_pending_experiment"]

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main autoresearch keep --root /Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos
{
  "decision": "keep",
  "commit": "c2ce6d3",
  "improved_best": true,
  "state": {
    "root": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos",
    "branch": "autoresearch/notebook-apr08-1252",
    "baseline_commit": "537c6e6",
    "baseline_bpb": 0.999,
    "best_commit": "c2ce6d3",
    "best_bpb": 0.998,
    "pending_experiment": null,
    "last_decision": "keep",
    "updated_at": "2026-04-08T10:53:51Z"
  },
  "experiment": {
    "success": true,
    "log_path": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/run.log",
    "timed_out": false,
    "return_code": 0,
    "val_bpb": 0.998,
    "peak_vram_mb": 1024.0,
    "training_seconds": 300.0,
    "total_seconds": 320.0,
    "mfu_percent": 10.0,
    "total_tokens_m": 1.0,
    "num_steps": 1,
    "num_params_m": 1.0,
   

('keep', False)

In [8]:
loop_result = run_json([
    PYTHON,
    "-m",
    "src.main",
    "autoresearch",
    "loop",
    "--root",
    str(AUTORESEARCH_ROOT),
    "--packet",
    str(packet_path),
    "--model",
    OLLAMA_MODEL,
    "--host",
    OLLAMA_HOST,
    "--iterations",
    "1",
    "--retry-limit",
    "1",
    "--trace",
])
final_status = run_json([
    PYTHON,
    "-m",
    "src.main",
    "autoresearch",
    "status",
    "--root",
    str(AUTORESEARCH_ROOT),
])
print("loop history entries:", len(loop_result["history"]))
print("memory_event_count:", final_status["memory_event_count"])
loop_decision = loop_result["history"][0]["decision"]["decision"] if loop_result["history"] else None
print("loop decision:", loop_decision)
if loop_decision == "no_pending_experiment":
    raise RuntimeError("Loop smoke test produced no pending experiment. The worker mutation was not repeatable.")
loop_decision

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main autoresearch loop --root /Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos --packet /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/notebooks/autoresearch_smoke_packet.json --model qwen2.5-coder:7b --host http://localhost:11434 --iterations 1 --retry-limit 1 --trace
{
  "setup": {
    "root": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos",
    "branch": "autoresearch/notebook-apr08-1252",
    "data_ready": true,
    "tokenizer_ready": true,
    "results_tsv_path": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/results.tsv",
    "results_tsv_initialized": false,
    "cache_dir": "/Users/wongdowling/.cache/autoresearch",
    "suggested_tag": "apr08"
  },
  "baseline": {
    "setup": {
      "root": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos",
      "branch": "autoresearch/notebook-apr08-1252",
  

'discard'

## Alternative Backends

The worker that edits `train.py` supports any model with an Ollama native API
**or** an OpenAI-compatible `/v1/chat/completions` endpoint.

| Backend | `--backend` flag | Typical server | Default port |
|---|---|---|---|
| Ollama native | `ollama` | `ollama serve` | 11434 |
| llama.cpp server | `openai-compat` | `llama-server` | 8080 |
| LM Studio | `openai-compat` | LM Studio app | 1234 |
| Ollama /v1 compat | `openai-compat` | `ollama serve` | 11434 |
| vLLM / TabbyAPI | `openai-compat` | varies | varies |


In [9]:
# Demonstrate backend selection: run with openai-compat pointing at Ollama /v1 endpoint
# (Ollama must be running; swap host/model for llama.cpp or LM Studio as needed)
# Note: --allow-any-branch is a loop-only flag; the run command uses the current branch as-is.

import json as _json

backend_raw = run_cmd([
    PYTHON, "-m", "src.main",
    "autoresearch", "run",
    "--root", str(AUTORESEARCH_ROOT),
    "--packet", str(packet_path),
    "--backend", "openai-compat",
    "--host", "http://localhost:11434",
    "--model", OLLAMA_MODEL,
], cwd=CLAW_ROOT, check=False)

try:
    backend_run_result = _json.loads(backend_raw.stdout)
    print("recommended_status:", backend_run_result.get("recommended_status", "?"))
except _json.JSONDecodeError:
    print("command stdout:", backend_raw.stdout[:400] or "(empty)")
    print("command stderr:", backend_raw.stderr[:400] or "(empty)")
    print("exit code:", backend_raw.returncode)


$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main autoresearch run --root /Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos --packet /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/notebooks/autoresearch_smoke_packet.json --backend openai-compat --host http://localhost:11434 --model qwen2.5-coder:7b
{
  "setup": {
    "root": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos",
    "branch": "autoresearch/notebook-apr08-1252",
    "data_ready": true,
    "tokenizer_ready": true,
    "results_tsv_path": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos/results.tsv",
    "results_tsv_initialized": false,
    "cache_dir": "/Users/wongdowling/.cache/autoresearch",
    "suggested_tag": "apr08"
  },
  "state": {
    "root": "/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos",
    "branch": "autoresearch/notebook-apr08-1252",
    "baseline_commit": "537c6e6",
    "b

## REST API Server

The `api-server` subcommand starts a local HTTP server so **any LLM with HTTP
tool-use** (or `curl`) can drive the research loop without Python imports.

```bash
python3 -m src.main api-server \
    --root ../../nodes/autoresearch-macos \
    --port 7331 \
    --backend openai-compat \
    --host http://localhost:8080
```

Then from any tool-calling LLM or shell:

```bash
# Check state
curl http://localhost:7331/status

# Run an experiment
curl -X POST http://localhost:7331/run \
  -H 'Content-Type: application/json' \
  -d '{"packet": {"objective": "...", "description": "..."}}'

# Accept or reject
curl -X POST http://localhost:7331/keep
curl -X POST http://localhost:7331/discard
```

The Rust `claw` binary can act as the manager by calling this API via its bash/HTTP tools,
keeping Claude's intelligence at the strategic level while the Python layer handles all state bookkeeping.


In [10]:
# Smoke-test the API server: start it in a background process, call /health, then stop it.
import subprocess, time, urllib.request, json as _json

api_proc = subprocess.Popen(
    [PYTHON, "-m", "src.main", "api-server",
     "--root", str(AUTORESEARCH_ROOT),
     "--port", "17331",          # use a non-default port to avoid conflicts
     "--backend", "ollama",
     "--host", OLLAMA_HOST,
     "--model", OLLAMA_MODEL],
    cwd=CLAW_ROOT,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(1.5)  # give the server a moment to bind

try:
    with urllib.request.urlopen("http://127.0.0.1:17331/health", timeout=3) as r:
        health = _json.loads(r.read())
    print("API /health:", health)
    with urllib.request.urlopen("http://127.0.0.1:17331/status", timeout=3) as r:
        status = _json.loads(r.read())
    print("API /status best_bpb:", status.get("state", {}).get("best_bpb"))
except Exception as e:
    print("API call failed:", e)
finally:
    api_proc.terminate()
    api_proc.wait()
    print("API server stopped.")


API /health: {'status': 'ok', 'root': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos'}
API /status best_bpb: 0.998
API server stopped.


## Success Criteria

This notebook is a successful control-plane smoke test when:

- `autoresearch setup` and `autoresearch status` return structured JSON
- `autoresearch isolate` moves the repo to an `autoresearch/<tag>` branch
- `autoresearch baseline` records or detects the baseline cleanly
- `autoresearch run` returns JSON with non-empty worker `tool_calls`
- `autoresearch status` shows a pending experiment after `run`
- `autoresearch keep` or `autoresearch discard` clears the pending experiment
- `autoresearch loop` returns a `history` entry and grows `experiment_memory.jsonl`

This proves baseline, keep/discard, loop, memory, and branch isolation wiring without waiting for a full training job.